# Sentiment analysis: Naive Bayes on Google Play Store reviews

Naive Bayes models are very useful when we want to analyze sentiment, classify texts into topics or recommendations, as the characteristics of these challenges meet the theoretical and methodological assumptions of the model very well.

In this project you will practice with a dataset to create a review classifier for the Google Play store.

## Step 1: Loading the dataset
The dataset can be found in this project folder under the name `playstore_reviews.csv`. You can load it into the code directly from the link:

`https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv`

In this dataset, you will find the following variables:

- **package_name**: Name of the mobile application (categorical)
- **review**: Comment about the mobile application (categorical)
- **polarity**: Class variable (0 or 1), being 0 a negative comment and 1 positive (categorical numeric)

## Step 2: Study of variables and their content
We have only 3 variables: 2 predictors and a dichotomous label. Of the two predictors, we are really only interested in the comment part, since the fact of classifying a comment as positive or negative will depend on its content, not on the application from which it was written. Therefore, the `package_name` variable should be removed.

When we work with text, it does not make sense to do a classic numeric EDA. Instead, we must preprocess the text:

1. Remove spaces and convert the text to lowercase:
   ```python
   df["column"] = df["column"].str.strip().str.lower()
   ```
2. Split the dataset into train and test: `X_train`, `X_test`, `y_train`, `y_test`.
3. Transform the text into a word count matrix:
   ```python
   vec_model = CountVectorizer(stop_words="english")
   X_train = vec_model.fit_transform(X_train).toarray()
   X_test = vec_model.transform(X_test).toarray()
   ```

## Step 3: Build a naive bayes model
Implement a model and choose the best option among:

- `GaussianNB`
- `MultinomialNB`
- `BernoulliNB`

Train all three and confirm whether the chosen model is correct.

## Step 4: Optimize the previous model
After choosing the best Naive Bayes option, try to optimize its results with a **Random Forest**, if possible.

## Step 5: Save the model
Store the model in the appropriate folder.

## Step 6: Explore other alternatives
Which other models of the ones we have studied could you use to try to overcome the results of a Naive Bayes? Argue this and train the model.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Step 1: Loading the dataset

In [2]:
# Step 1: Load the dataset
url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"

df = pd.read_csv(url)
print("Shape:", df.shape)
df.head()

Shape: (891, 3)


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


## Step 2: Study of variables and text preprocessing

In [3]:
# Step 2: Keep only the text and the label
# We drop 'package_name' because sentiment depends on the review text, not the app name.
df = df.drop(columns=["package_name"])

# Basic text cleaning
df["review"] = df["review"].astype(str).str.strip().str.lower()

# Split predictors and target
X = df["review"]
y = df["polarity"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train_text.shape[0])
print("Test size:", X_test_text.shape[0])
print()
print("Train class balance:")
print(y_train.value_counts(normalize=True).sort_index())

Train size: 712
Test size: 179

Train class balance:
polarity
0    0.655899
1    0.344101
Name: proportion, dtype: float64


### Text to numeric features (CountVectorizer)

In [4]:
# Convert text into numeric features (word counts)
# max_features limits vocabulary size to keep the matrix small and memory-friendly.
vec_model = CountVectorizer(stop_words="english", max_features=5000)

X_train_counts = vec_model.fit_transform(X_train_text)
X_test_counts = vec_model.transform(X_test_text)

print("X_train_counts shape:", X_train_counts.shape)
print("X_test_counts shape:", X_test_counts.shape)

X_train_counts shape: (712, 3272)
X_test_counts shape: (179, 3272)


## Step 3: Build a Naive Bayes model (compare 3 options)

In [5]:
# Step 3: Train and compare Naive Bayes implementations

results = []

# 1) MultinomialNB (usually a good fit for word counts)
mnb = MultinomialNB()
mnb.fit(X_train_counts, y_train)
pred_mnb = mnb.predict(X_test_counts)

results.append({
    "model": "MultinomialNB",
    "accuracy": accuracy_score(y_test, pred_mnb)
})

# 2) BernoulliNB (often used for binary features)
# We convert counts to binary (0 or 1).
X_train_binary = (X_train_counts > 0).astype(int)
X_test_binary = (X_test_counts > 0).astype(int)

bnb = BernoulliNB()
bnb.fit(X_train_binary, y_train)
pred_bnb = bnb.predict(X_test_binary)

results.append({
    "model": "BernoulliNB",
    "accuracy": accuracy_score(y_test, pred_bnb)
})

# 3) GaussianNB (expects dense data)
# We convert the sparse matrix to a dense array. This can use more memory.
gnb = GaussianNB()
gnb.fit(X_train_counts.toarray(), y_train)
pred_gnb = gnb.predict(X_test_counts.toarray())

results.append({
    "model": "GaussianNB",
    "accuracy": accuracy_score(y_test, pred_gnb)
})

results_df = pd.DataFrame(results).sort_values(by="accuracy", ascending=False)
results_df

,model,accuracy
0,MultinomialNB,0.854749
2,GaussianNB,0.815642
1,BernoulliNB,0.782123


In [7]:
# Show detailed metrics for the best Naive Bayes model (by accuracy)

best_nb_name = results_df.iloc[0]["model"]
print("Best Naive Bayes model:", best_nb_name)

if best_nb_name == "MultinomialNB":
    best_nb_model = mnb
    best_pred = pred_mnb
elif best_nb_name == "BernoulliNB":
    best_nb_model = bnb
    best_pred = pred_bnb
else:
    best_nb_model = gnb
    best_pred = pred_gnb

print("Accuracy:", accuracy_score(y_test, best_pred))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, best_pred))
print()
print("Classification report:")
print(classification_report(y_test, best_pred))

Best Naive Bayes model: MultinomialNB
Accuracy: 0.8547486033519553

Confusion matrix:
[[112   5]
 [ 21  41]]

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.96      0.90       117
           1       0.89      0.66      0.76        62

    accuracy                           0.85       179
   macro avg       0.87      0.81      0.83       179
weighted avg       0.86      0.85      0.85       179



## Step 4: Optimize the previous model (try Random Forest)

In [8]:
# Step 4: Try to improve with a Random Forest (if possible)
# Random forests are not text-specialists, but we can still try them on the vectorized features.

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=None
)

# RandomForestClassifier usually expects dense arrays, so we convert.
X_train_dense = X_train_counts.toarray()
X_test_dense = X_test_counts.toarray()

rf.fit(X_train_dense, y_train)
pred_rf = rf.predict(X_test_dense)

print("Random Forest accuracy:", accuracy_score(y_test, pred_rf))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, pred_rf))
print()
print("Classification report:")
print(classification_report(y_test, pred_rf))

Random Forest accuracy: 0.8212290502793296

Confusion matrix:
[[105  12]
 [ 20  42]]

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87       117
           1       0.78      0.68      0.72        62

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.82       179



## Step 5: Save the model

In [9]:
# Step 5: Save the best model
# For saving, it is useful to store a pipeline: vectorizer + classifier together.
# Here we save the best-performing model among: best NB vs Random Forest vs alternative model (next step).

os.makedirs("models", exist_ok=True)

## Step 6: Explore other alternatives

A few models we have studied that often beat Naive Bayes on text classification are:

- **Logistic Regression** (linear model, strong baseline for text)
- **Support Vector Machines (Linear SVM)** (also strong for text, if available in your module)
- **Gradient Boosting** (can work, but usually not the first choice for high-dimensional sparse text)

Below we train **Logistic Regression** using **TF-IDF** features, which is a common and strong approach.

In [10]:
# TF-IDF features + Logistic Regression (common strong baseline)

tfidf = TfidfVectorizer(stop_words="english", max_features=20000)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
pred_lr = lr.predict(X_test_tfidf)

print("Logistic Regression accuracy:", accuracy_score(y_test, pred_lr))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, pred_lr))
print()
print("Classification report:")
print(classification_report(y_test, pred_lr))

Logistic Regression accuracy: 0.7821229050279329

Confusion matrix:
[[112   5]
 [ 34  28]]

Classification report:
              precision    recall  f1-score   support

           0       0.77      0.96      0.85       117
           1       0.85      0.45      0.59        62

    accuracy                           0.78       179
   macro avg       0.81      0.70      0.72       179
weighted avg       0.80      0.78      0.76       179



### Final comparison (Naive Bayes vs Random Forest vs Alternative)

In [12]:
# Pick the best model by accuracy and save it

final_results = [
    {"model": best_nb_name, "accuracy": accuracy_score(y_test, best_pred)},
    {"model": "RandomForest", "accuracy": accuracy_score(y_test, pred_rf)},
    {"model": "LogisticRegression_TFIDF", "accuracy": accuracy_score(y_test, pred_lr)},
]

final_results_df = pd.DataFrame(final_results).sort_values(by="accuracy", ascending=False)
final_results_df

,model,accuracy
0,MultinomialNB,0.854749
1,RandomForest,0.821229
2,LogisticRegression_TFIDF,0.782123


In [13]:
best_final_name = final_results_df.iloc[0]["model"]
print("Best final model:", best_final_name)

if best_final_name == "LogisticRegression_TFIDF":
    # Save TF-IDF + Logistic Regression as a single object
    model_to_save = {
        "vectorizer": tfidf,
        "model": lr
    }
    save_path = "models/playstore_sentiment_model.joblib"
    joblib.dump(model_to_save, save_path)

elif best_final_name == "RandomForest":
    model_to_save = {
        "vectorizer": vec_model,
        "model": rf
    }
    save_path = "models/playstore_sentiment_model.joblib"
    joblib.dump(model_to_save, save_path)

else:
    # Best Naive Bayes (Multinomial, Bernoulli, or Gaussian)
    # Note: BernoulliNB used binary features; we keep the count vectorizer and binarize at prediction time if needed.
    model_to_save = {
        "vectorizer": vec_model,
        "model": best_nb_model,
        "note": "If the model is BernoulliNB, convert counts to binary before predicting."
    }
    save_path = "models/playstore_sentiment_model.joblib"
    joblib.dump(model_to_save, save_path)

print("Saved model at:", save_path)

Best final model: MultinomialNB
Saved model at: models/playstore_sentiment_model.joblib
